# BraTS Subset

Extract required BraTS files with `neuroseg_2d.data`, select the configured fraction of each grade, then archive only FLAIR and segmentation pairs for faster transfer.

## 1. Imports and configuration

In [ ]:
import random
from pathlib import Path
from zipfile import ZIP_DEFLATED, BadZipFile, ZipFile

from neuroseg_2d.data import discover, extract_dataset

# Define source, output, and deterministic selection settings.
SOURCE_ARCHIVE = (Path.home() / "datasets/brats_2019.zip").resolve()
DATA_DIR = SOURCE_ARCHIVE.with_suffix("")
FRACTION = 0.1
SUBSET_ARCHIVE = SOURCE_ARCHIVE.with_name(f"{SOURCE_ARCHIVE.stem}_{FRACTION:g}.zip")
SEED = 42
GRADES = ("HGG", "LGG")

if not SOURCE_ARCHIVE.is_file():
    raise FileNotFoundError(f"BraTS archive not found: {SOURCE_ARCHIVE}")

## 2. Extract and discover patients

Extraction is safe to repeat: complete files stay unchanged and missing or incomplete files are replaced.

In [ ]:
# Extract reusable FLAIR and segmentation pairs.
try:
    data_dir = extract_dataset(SOURCE_ARCHIVE, DATA_DIR)
except BadZipFile as error:
    raise ValueError(f"Invalid ZIP archive: {SOURCE_ARCHIVE}") from error

# Group discovered patients by grade in stable order.
patients = discover(data_dir)
patients_by_grade = {
    grade: sorted(
        (patient for patient in patients if patient.grade == grade),
        key=lambda patient: patient.id,
    )
    for grade in GRADES
}

print(f"Source: {SOURCE_ARCHIVE}")
print(f"Extracted data: {data_dir}")
print(f"Patients: {len(patients)}")

## 3. Select a fraction from each grade

Sorted patients and fixed seeds keep selection stable across runs.

In [ ]:
# Sample each grade independently with the fixed seed.
selected_by_grade = {}
for grade in GRADES:
    grade_patients = patients_by_grade[grade]
    sample_size = round(FRACTION * len(grade_patients))
    if sample_size == 0:
        raise ValueError(f"Not enough {grade} patients for fraction {FRACTION}")

    selected = sorted(
        random.Random(SEED).sample(grade_patients, sample_size),
        key=lambda patient: patient.id,
    )
    selected_by_grade[grade] = selected

# Flatten grade groups in declared grade order.
selected_patients = [patient for grade in GRADES for patient in selected_by_grade[grade]]

## 4. Archive selected FLAIR and segmentation files

Write a temporary ZIP, verify exact contents, then publish it next to the full archive.

In [ ]:
# Collect selected paths and expected ZIP member names.
selected_files = [
    src_file for patient in selected_patients for src_file in (patient.flair_file, patient.seg_file)
]
selected_members = sorted(src_file.relative_to(data_dir).as_posix() for src_file in selected_files)
temporary_archive = SUBSET_ARCHIVE.with_name(f".{SUBSET_ARCHIVE.name}.tmp")
SUBSET_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)

# Write through a temporary archive to avoid publishing incomplete output.
with ZipFile(temporary_archive, "w", compression=ZIP_DEFLATED) as archive:
    for src_file in selected_files:
        archive.write(
            src_file,
            arcname=src_file.relative_to(data_dir).as_posix(),
        )

# Verify exact contents before publishing the final archive.
with ZipFile(temporary_archive) as archive:
    actual_members = sorted(archive.namelist())
    assert archive.testzip() is None

assert actual_members == selected_members
assert len(actual_members) == 2 * len(selected_patients)
temporary_archive.replace(SUBSET_ARCHIVE)

# Print reproducible selection summary.
print(f"Subset archive: {SUBSET_ARCHIVE}")
print(f"Fraction: {FRACTION}, seed: {SEED}")
for grade in GRADES:
    selected_ids = [patient.id for patient in selected_by_grade[grade]]
    print(f"{grade}: {len(selected_ids)} patients")
    print(f"  {', '.join(selected_ids)}")